# SemRel — Model Evaluation & Comparison
**COS 760 Group Project**  
Loads predictions from all models and produces a unified comparison across:
- **AfriBERT** (XLM-RoBERTa-base fine-tuned, notebook 04)
- **Embedding cosine baseline** (multilingual sentence-transformer, from notebook 03)
- **TF-IDF cosine baseline** (from notebook 03)

Metrics: Spearman ρ, Pearson r, MSE, RMSE — on the **test** split.

## 1. Install & Import Dependencies

In [ ]:
!pip install pandas numpy matplotlib seaborn scipy scikit-learn -q

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, pearsonr
from sklearn.metrics import mean_squared_error

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.size']  = 11

DATA_DIR     = './cleaned_data'
FEATURES_DIR = './features'
AFRIBERT_DIR = './model_outputs/afribert_best'
OUTPUT_DIR   = './evaluation_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Setup complete.')


## 2. Load Gold Labels & Model Predictions

In [ ]:
# ── Gold labels ──────────────────────────────────────────────────────────────
test_df = pd.read_csv(os.path.join(DATA_DIR, 'afr_test.csv'))
gold    = test_df['score'].values
print(f'Test set: {len(test_df)} pairs')
display(test_df.head(3))


In [ ]:
# ── AfriBERT predictions ─────────────────────────────────────────────────────
afribert_pred_path = os.path.join(AFRIBERT_DIR, 'test_predictions.csv')
if os.path.exists(afribert_pred_path):
    afribert_df    = pd.read_csv(afribert_pred_path)
    afribert_preds = afribert_df['afribert_pred'].values
    print(f'AfriBERT predictions loaded: {len(afribert_preds)} rows')
else:
    print(f'WARNING: {afribert_pred_path} not found — run 04_AfriBERT.ipynb first.')
    afribert_preds = None


In [ ]:
# ── Baseline predictions (embedding cosine & TF-IDF cosine) from feature CSV ─
feat_path = os.path.join(FEATURES_DIR, 'afr_test_features.csv')
if os.path.exists(feat_path):
    feat_df = pd.read_csv(feat_path)
    emb_preds   = feat_df['embedding_cosine'].values
    tfidf_preds = feat_df['tfidf_cosine'].values
    print(f'Feature-based predictions loaded: {len(feat_df)} rows')
    print(f'  Columns: {feat_df.columns.tolist()}')
else:
    print(f'WARNING: {feat_path} not found — run 03_feature_engineering.ipynb first.')
    emb_preds   = None
    tfidf_preds = None


## 3. Metric Helper

In [ ]:
def compute_metrics(preds: np.ndarray, gold: np.ndarray, name: str) -> dict:
    """Return a dict of evaluation metrics for one model."""
    mse      = float(np.mean((preds - gold) ** 2))
    rmse     = float(np.sqrt(mse))
    pearson  = float(pearsonr(preds, gold)[0])
    spearman = float(spearmanr(preds, gold)[0])
    return {
        'Model':    name,
        'Spearman': round(spearman, 4),
        'Pearson':  round(pearson, 4),
        'MSE':      round(mse, 4),
        'RMSE':     round(rmse, 4),
    }


## 4. Compute All Metrics

In [ ]:
results = []

if afribert_preds is not None:
    results.append(compute_metrics(afribert_preds, gold, 'AfriBERT (XLM-R-base)'))

if emb_preds is not None:
    results.append(compute_metrics(emb_preds, gold, 'Embedding Cosine Baseline'))

if tfidf_preds is not None:
    results.append(compute_metrics(tfidf_preds, gold, 'TF-IDF Cosine Baseline'))

results_df = pd.DataFrame(results).set_index('Model')

print('Test-set results:')
display(results_df.style.highlight_max(axis=0, subset=['Spearman', 'Pearson'], color='#b7e4c7')
                        .highlight_min(axis=0, subset=['MSE', 'RMSE'], color='#b7e4c7')
                        .format('{:.4f}'))


## 5. Bar Charts — Metric Comparison

In [ ]:
if results:
    models = [r['Model'] for r in results]
    metrics = ['Spearman', 'Pearson', 'MSE', 'RMSE']
    colors  = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

    fig, axes = plt.subplots(1, 4, figsize=(18, 5))

    for ax, metric, color in zip(axes, metrics, colors):
        vals = [r[metric] for r in results]
        bars = ax.bar(models, vals, color=color, edgecolor='white', alpha=0.85)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.002,
                    f'{val:.4f}', ha='center', va='bottom', fontsize=8)
        ax.set_title(metric)
        ax.set_ylabel(metric)
        ax.tick_params(axis='x', rotation=20)

    fig.suptitle('Model Comparison — Afrikaans SemRel2024 Test Set', fontsize=13)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'metric_comparison_bars.png'), dpi=150, bbox_inches='tight')
    plt.show()


## 6. Scatter Plots — Predicted vs Gold

In [ ]:
models_to_plot = []
if afribert_preds is not None:
    models_to_plot.append(('AfriBERT (XLM-R-base)', afribert_preds, '#4C72B0'))
if emb_preds is not None:
    models_to_plot.append(('Embedding Cosine', emb_preds, '#DD8452'))
if tfidf_preds is not None:
    models_to_plot.append(('TF-IDF Cosine', tfidf_preds, '#55A868'))

n = len(models_to_plot)
if n > 0:
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
    if n == 1: axes = [axes]

    for ax, (name, preds, color) in zip(axes, models_to_plot):
        ax.scatter(gold, preds, alpha=0.3, s=10, color=color)
        ax.plot([0, 1], [0, 1], 'r--', linewidth=1.2, label='Perfect')
        spear = spearmanr(preds, gold)[0]
        pear  = pearsonr(preds, gold)[0]
        ax.set_title(f'{name}\nSpearman={spear:.4f}  Pearson={pear:.4f}', fontsize=10)
        ax.set_xlabel('Gold score')
        ax.set_ylabel('Predicted score')
        ax.legend(fontsize=8)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)

    fig.suptitle('Predicted vs Gold — Afrikaans Test Set', fontsize=13)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'scatter_all_models.png'), dpi=150, bbox_inches='tight')
    plt.show()


## 7. Residual Analysis

In [ ]:
if models_to_plot:
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 4))
    if n == 1: axes = [axes]

    for ax, (name, preds, color) in zip(axes, models_to_plot):
        residuals = preds - gold
        ax.hist(residuals, bins=30, color=color, edgecolor='white', alpha=0.85)
        ax.axvline(0, color='red', linestyle='--', linewidth=1.2)
        ax.set_title(f'{name}\nMean residual={residuals.mean():.4f}  Std={residuals.std():.4f}', fontsize=9)
        ax.set_xlabel('Residual (pred − gold)')
        ax.set_ylabel('Count')

    fig.suptitle('Residual Distributions — Afrikaans Test Set', fontsize=13)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'residuals_all_models.png'), dpi=150, bbox_inches='tight')
    plt.show()


## 8. Error Analysis — Worst Predictions
Inspect pairs where the best model is furthest from the gold label.

In [ ]:
if afribert_preds is not None:
    error_df = test_df[['sentence1', 'sentence2', 'score']].copy()
    error_df.rename(columns={'score': 'gold'}, inplace=True)
    error_df['afribert_pred'] = afribert_preds
    error_df['abs_error']    = (error_df['afribert_pred'] - error_df['gold']).abs()
    error_df = error_df.sort_values('abs_error', ascending=False)

    print('Top 10 worst predictions (AfriBERT):')
    display(error_df.head(10)[['sentence1', 'sentence2', 'gold', 'afribert_pred', 'abs_error']])

    # Save error analysis
    error_df.to_csv(os.path.join(OUTPUT_DIR, 'error_analysis.csv'), index=False)
    print(f'\nFull error analysis saved to {OUTPUT_DIR}/error_analysis.csv')
else:
    print('AfriBERT predictions not available for error analysis.')


## 9. Score Bucket Performance
Break test pairs into Low / Medium / High relatedness and measure Spearman per bucket.

In [ ]:
def spearman_by_bucket(preds, gold_scores, buckets):
    """Return Spearman rho for each bucket label."""
    bucket_results = {}
    for label, mask in buckets.items():
        if mask.sum() < 5:
            bucket_results[label] = np.nan
            continue
        bucket_results[label] = round(float(spearmanr(preds[mask], gold_scores[mask])[0]), 4)
    return bucket_results

buckets = {
    'Low  (<0.33)':   gold < 0.33,
    'Mid  (0.33–0.67)': (gold >= 0.33) & (gold < 0.67),
    'High (≥0.67)':   gold >= 0.67,
}

bucket_rows = []
for name, preds, _ in models_to_plot:
    row = {'Model': name}
    row.update(spearman_by_bucket(preds, gold, buckets))
    bucket_rows.append(row)

if bucket_rows:
    bucket_df = pd.DataFrame(bucket_rows).set_index('Model')
    print('Spearman ρ by score bucket:')
    display(bucket_df)

    # Heatmap
    fig, ax = plt.subplots(figsize=(8, max(3, len(bucket_rows) * 0.8 + 1)))
    sns.heatmap(bucket_df.astype(float), annot=True, fmt='.4f', cmap='YlGn',
                linewidths=0.5, ax=ax, vmin=0, vmax=1)
    ax.set_title('Spearman ρ by Score Bucket — Afrikaans Test Set')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'bucket_heatmap.png'), dpi=150, bbox_inches='tight')
    plt.show()


## 10. Final Summary Table & Export

In [ ]:
print('=' * 60)
print('FINAL RESULTS — Afrikaans SemRel2024 Test Set')
print('=' * 60)
display(results_df)

results_df.to_csv(os.path.join(OUTPUT_DIR, 'final_results.csv'))
print(f'\nResults saved to {OUTPUT_DIR}/final_results.csv')

# Best model
if not results_df.empty:
    best_model = results_df['Spearman'].idxmax()
    print(f'\nBest model by Spearman ρ: {best_model}  '
          f'(ρ = {results_df.loc[best_model, "Spearman"]:.4f})')
